# 02 — Model Training
Fine-tune YOLOv11 for Egyptian ID field detection with MLflow tracking

In [ ]:
import sys
sys.path.insert(0, '..')
import mlflow
from ultralytics import YOLO
from src.data.validate import DatasetValidator

## Step 1: Validate Dataset

In [ ]:
validator = DatasetValidator('../Egyptain-Person-ID-1')
is_valid = validator.run()
report = validator.report()
print('Valid:', is_valid)
for issue in report['issues']:
    print('  Issue:', issue)
print('Stats:', report['stats'])

## Step 2: Configure MLflow

In [ ]:
mlflow.set_tracking_uri('../mlruns')
mlflow.set_experiment('arabic-ocr-detection')
print('MLflow tracking URI:', mlflow.get_tracking_uri())
# Run: mlflow ui --backend-store-uri ./mlruns to view experiments

## Step 3: Train — Quick Sanity Run (5 epochs)

In [ ]:
# Quick sanity check with 5 epochs. Change epochs=100 for full training.
with mlflow.start_run(tags={'notebook': 'training', 'purpose': 'sanity-check'}) as run:
    mlflow.log_params({'model': 'yolo11n', 'epochs': 5, 'batch': 8, 'img_size': 640})
    
    model = YOLO('yolo11n.pt')
    results = model.train(
        data='../Egyptain-Person-ID-1/data.yaml',
        epochs=5,
        batch=8,
        imgsz=640,
        device='cpu',  # change to '0' for GPU
        project='../runs/train',
        name='sanity_check',
        exist_ok=True,
    )
    
    if hasattr(results, 'results_dict'):
        m = results.results_dict
        mlflow.log_metrics({
            'mAP50': m.get('metrics/mAP50(B)', 0),
            'mAP50_95': m.get('metrics/mAP50-95(B)', 0),
        })
    
    print(f'Run ID: {run.info.run_id}')

## Step 4: Full Training via CLI
Run `make train` or `python src/training/train.py` for production training.

In [ ]:
# View all experiments
client = mlflow.tracking.MlflowClient('../mlruns')
experiments = client.search_experiments()
for exp in experiments:
    runs = client.search_runs(exp.experiment_id, order_by=['metrics.mAP50 DESC'], max_results=5)
    print(f"\nExperiment: {exp.name}")
    for r in runs:
        mAP = r.data.metrics.get('mAP50', 'N/A')
        print(f"  Run {r.info.run_id[:8]}: mAP50={mAP}")